# numpy-基本功能

## 教學目標

這份教學的目標是介紹基本的 numpy 功能，並學習快速操作大量數值型態的資料。

## 適用對象

適用於有程式基礎，且擁有 python 基礎的學生。

若沒有先學過 Python，建議先補充 Python 入門語法。

## 原教材的執行時間與測試環境

以下是原教材的測試記錄，實際執行時間會因內容與環境而不同。

本教學全部執行時間約為 1.7527921199798584 秒。

|測試環境|名稱|
|-|-|
|主機板|X570 AORUS ELITE|
|處理器|AMD Ryzen 7 3700X 8-Core Processor|
|記憶體|Kingston KHX3200C16D4/16GX|
|硬碟|Seagate ST1000DM003-1ER1|
|顯示卡|GeForce RTX 2080|
|作業系統|Ubuntu 18.04 LTS|

套件安裝與教材導覽請參考 [使用說明](./README.md)。

## 大綱

- 簡介與安裝
- 陣列 shape 與資料取值
- 整批運算、axis 與 broadcasting
- 矩陣乘法與 reshape
- 練習

## 簡介

根據 [Numpy 官方網站](https://numpy.org/)（v1.18）：

> NumPy is the fundamental package for scientific computing with Python.
> 
> NumPy 是設計用於進行科學計算的 Python 工具
> 
> It contains among other things:
>
> 包含了以下內容:
>
> - a powerful N-dimensional array object
>
> N 維度的矩陣運算功能
>
> - sophisticated (broadcasting) functions
>
> 方便的運算傳播功能
>
> - tools for integrating C/C++ and Fortran code
>
> 能夠跟 C/C++/Fortran 程式碼整合的工具
>
> - useful linear algebra, Fourier transform, and random number capabilities
>
> 包含線性代數、傅利葉轉換、隨機變數的功能

## 安裝

透過 `pip` 安裝 `numpy`：

```sh
pip install numpy
```

In [12]:
import numpy as np

## 1. 先看 shape：樣本 × 特徵

ML 常用 `X.shape == (樣本數, 特徵數)`，`y` 是每筆樣本對應的答案。
`dtype` 用來確認數值型態；本篇不逐一介紹整數位元範圍。

In [13]:
X = np.array([[18, 90], [25, 75], [40, 60], [32, 85]], dtype=float)
y = np.array([0, 1, 1, 0])
print("X:", X.shape, "y:", y.shape, "dtype:", X.dtype)

X: (4, 2) y: (4,) dtype: float64


## 2. 取資料：列是樣本，欄是特徵

`X[列, 欄]`；切片不含結束位置。保留單一特徵時，注意 `(n,)` 與 `(n, 1)` 的差別。

In [14]:
print("前兩筆樣本：\n", X[:2])
print("第一個特徵，一維：", X[:, 0].shape)
print("第一個特徵，二維：", X[:, :1].shape)
print("第一個特徵 >= 30 的樣本：\n", X[X[:, 0] >= 30])

前兩筆樣本：
 [[18. 90.]
 [25. 75.]]
第一個特徵，一維： (4,)
第一個特徵，二維： (4, 1)
第一個特徵 >= 30 的樣本：
 [[40. 60.]
 [32. 85.]]


## 3. 整批運算與 axis

陣列可以整批計算，不必逐筆寫迴圈。
對二維資料，`axis=0` 跨樣本、得到每個特徵的統計值；`axis=1` 跨特徵、得到每筆樣本的統計值。
混合不同單位的特徵時，跨特徵平均未必有意義。

In [15]:
print("每個特徵的平均：", X.mean(axis=0))
print("每個特徵的標準差：", X.std(axis=0))
scores = np.array([[80, 90], [60, 70]])
print("每位學生的平均成績：", scores.mean(axis=1))
print("成績加 5 分：\n", scores + 5)

每個特徵的平均： [28.75 77.5 ]
每個特徵的標準差： [ 8.16624149 11.45643924]
每位學生的平均成績： [85. 65.]
成績加 5 分：
 [[85 95]
 [65 75]]


### 組合條件並保持標籤對應

判斷式會產生布林陣列。用 `&` 組合兩個條件時，每個條件都要加上括號；
用相同條件篩選樣本和標籤，就能保持兩者的順序與對應關係。
以下整理自原教材的 NumPy 判斷式取值與 pandas 多條件篩選範例。

In [16]:
mask_demo = (X[:, 0] >= 20) & (X[:, 1] >= 80)
print('符合條件：', mask_demo)
print('篩選後的樣本：', X[mask_demo])
print('對應的標籤：', y[mask_demo])

符合條件： [False False False  True]
篩選後的樣本： [[32. 85.]]
對應的標籤： [0]


對應練習：[第 1 題](./exercises.ipynb#exercise-1)。


## 4. Broadcasting：每個特徵用自己的統計值

`(4, 2)` 減去 `(2,)`，會把兩個平均值套用到每一筆樣本。
判斷 shape 是否相容：從最右邊比起，大小要相等，或其中一邊為 1；缺少的維度視為 1。

以下示範每個特徵減平均、除標準差。實際 ML 要先切分資料，統計值只從訓練集估計，
驗證／測試集沿用它們，避免資料洩漏。

In [17]:
mean = X.mean(axis=0)
std = X.std(axis=0)
X_scaled = (X - mean) / np.where(std == 0, 1, std)
print("輸出 shape：", X_scaled.shape)
print("標準化後每個特徵的平均：", X_scaled.mean(axis=0))

輸出 shape： (4, 2)
標準化後每個特徵的平均： [4.16333634e-17 2.77555756e-17]


## 5. 矩陣乘法：讀懂線性模型

`*` 是對應位置相乘；`@` 是矩陣乘法。
線性模型 `X @ w + b` 將每筆樣本的特徵加權後，得到一個預測值。
`(4, 2) @ (2,) → (4,)`，中間的特徵數必須一致。

In [18]:
w = np.array([0.2, -0.1])
b = 1.0
pred = X @ w + b
print("逐元素相乘的 shape：", (X * w).shape)
print("預測：", pred, "shape：", pred.shape)

逐元素相乘的 shape： (4, 2)
預測： [-4.4 -1.5  3.  -1.1] shape： (4,)


### 用布林陣列檢查分類結果

比較預測與標籤可得到每筆是否正確的布林值；布林值的平均數是正確比例。
把布林條件交給 `np.array(..., dtype=int)` 可轉成 0／1，
再用條件篩選檢查個別錯誤。這裡沿用原教材的判斷式取值與平均數運算。

In [19]:
actual_demo = np.array([1, 0, 1])
pred_demo = np.array([1, 1, 0])
print('是否正確：', pred_demo == actual_demo)
print('正確比例：', np.mean(pred_demo == actual_demo))
print('把布林值轉成 0/1：', np.array(pred_demo > 0, dtype=int))
print('漏判的標籤：', actual_demo[(actual_demo == 1) & (pred_demo == 0)])

是否正確： [ True False False]
正確比例： 0.3333333333333333
把布林值轉成 0/1： [1 1 0]
漏判的標籤： [1]


對應練習：[第 2 題](./exercises.ipynb#exercise-2)。


## 6. reshape 與可重現的範例

單一特徵常需要用 `reshape(-1, 1)` 轉成二維；`-1` 表示自動推算大小，元素總數不變。
`reshape` 不會替你決定樣本和特徵的意義。固定亂數種子便於比較結果。

In [20]:
age = X[:, 0]
print("轉換前後：", age.shape, age.reshape(-1, 1).shape)
rng = np.random.default_rng(42)
print("模擬資料：\n", rng.normal(size=(3, 2)))

轉換前後： (4,) (4, 1)
模擬資料：
 [[ 0.30471708 -1.03998411]
 [ 0.7504512   0.94056472]
 [-1.95103519 -1.30217951]]


## 用 AI 協作時，先驗收這三件事

- 輸入和輸出的 shape 是否符合任務？
- 統計值是按特徵算，還是按樣本算？
- 使用的是逐元素運算，還是矩陣乘法？

接下來保留兩個 ML 練習。可請 AI 協助實作，但需解釋公式與輸出。
softmax 的輸出應非負且總和約為 1；Linear Layer 的輸出維度應是 `d_out`，經 ReLU 後不應有負值。

## 練習：固定輸出的程式填空


### 練習 1：固定輸入的 Softmax（程式填空）

補完穩定 Softmax：先減去最大值，再取指數並除以總和。輸出保留六位小數。

請補上 `____`，保留固定資料與所有 print，不可直接寫死預期結果。NumPy 題請先匯入 `numpy as np`；pandas 題沿用教材讀入、尚未填補的原始 `df`。

**預期輸出（逐行比對）：**

```text
probabilities: [0.166667, 0.333333, 0.5]
sum: 1.0
```


In [ ]:
logits = np.log(np.array([1., 2., 3.]))
shifted = logits - logits.____()
exp_values = np.____(shifted)
probabilities = exp_values / exp_values.____()
print('probabilities:', np.round(probabilities, 6).tolist())
print('sum:', round(float(probabilities.sum()), 6))


### 練習 2：Linear Layer + ReLU（程式填空）

以矩陣乘法算出 z = W @ x + b，再把負值改成 0。請保留二維 shape。

請補上 `____`，保留固定資料與所有 print，不可直接寫死預期結果。NumPy 題請先匯入 `numpy as np`；pandas 題沿用教材讀入、尚未填補的原始 `df`。

**預期輸出（逐行比對）：**

```text
linear: [[4.0], [-4.0], [-3.0]]
relu: [[4.0], [0.0], [0.0]]
shape: (3, 1)
```


In [ ]:
x = np.array([[1.], [-2.]])
W = np.array([[2., -1.], [-1., 2.], [0., 0.]])
bias = np.array([[0.], [1.], [-3.]])
z = ____ @ ____ + ____
activation = z.copy()
activation[activation < ____] = ____
print('linear:', z.tolist())
print('relu:', activation.tolist())
print('shape:', activation.shape)
